In [1]:
#pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd

In [5]:
xml_path = Path("data/climate_dataset_datacite.xml")

print("XML path:", xml_path)
print("File exists:", xml_path.exists())

if not xml_path.exists():
    raise FileNotFoundError(f"Could not find: {xml_path}")

XML path: data\climate_dataset_datacite.xml
File exists: True


In [6]:
raw_xml = xml_path.read_text(encoding="utf-8")
print(raw_xml[:1500])

<?xml version="1.0" encoding="UTF-8"?>
<resource
  xmlns="http://datacite.org/schema/kernel-4"
  xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
  xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4/metadata.xsd">

  <identifier identifierType="DOI">10.1234/gcd-2025</identifier>

  <creators>
    <creator>
      <creatorName>Global Climate Data Team</creatorName>
    </creator>
  </creators>

  <titles>
    <title>2025 Global Climate Data</title>
  </titles>

  <publisher>Open Research Repository (Teaching)</publisher>

  <publicationYear>2025</publicationYear>

  <resourceType resourceTypeGeneral="Dataset">Dataset</resourceType>

  <descriptions>
    <description descriptionType="Abstract">
      A small teaching dataset containing fictional annual climate summaries.
      Variables include average temperature, rainfall, and CO2 emissions for two fictional countries
      (Alandia and Borealia) from 2021 to 2023.
    </description>
  

In [7]:
tree = ET.parse(xml_path)
root = tree.getroot()

print("Root tag:", root.tag)

Root tag: {http://datacite.org/schema/kernel-4}resource


In [8]:
def parse_datacite_xml(xml_file: Path) -> dict:
    root = ET.parse(xml_file).getroot()

    identifier = None
    creators = []
    title = None
    publisher = None
    publication_year = None
    resource_type = None
    description = None

    for el in root.iter():
        name = local_name(el.tag)
        text = (el.text or "").strip()

        if name == "identifier" and text and identifier is None:
            identifier = text
        elif name == "creatorName" and text:
            creators.append(text)
        elif name == "title" and text and title is None:
            title = text
        elif name == "publisher" and text and publisher is None:
            publisher = text
        elif name == "publicationYear" and text and publication_year is None:
            publication_year = text
        elif name == "resourceType" and text and resource_type is None:
            resource_type = text
        elif name == "description" and text and description is None:
            description = text

    return {
        "identifier": identifier,
        "creators": creators,
        "title": title,
        "publisher": publisher,
        "publicationYear": publication_year,
        "resourceType": resource_type,
        "description": description,
    }

In [9]:
datacite_meta = parse_datacite_xml(xml_path)
datacite_meta

{'identifier': '10.1234/gcd-2025',
 'creators': ['Global Climate Data Team'],
 'title': '2025 Global Climate Data',
 'publisher': 'Open Research Repository (Teaching)',
 'publicationYear': '2025',
 'resourceType': 'Dataset',
 'description': 'A small teaching dataset containing fictional annual climate summaries.\n      Variables include average temperature, rainfall, and CO2 emissions for two fictional countries\n      (Alandia and Borealia) from 2021 to 2023.'}

In [10]:
rows = [
    ("identifier", datacite_meta.get("identifier")),
    ("creators", "; ".join(datacite_meta.get("creators", []))),
    ("title", datacite_meta.get("title")),
    ("publisher", datacite_meta.get("publisher")),
    ("publicationYear", datacite_meta.get("publicationYear")),
    ("resourceType", datacite_meta.get("resourceType")),
    ("description", datacite_meta.get("description")),
]

table = pd.DataFrame(rows, columns=["field", "value"])
table

,field,value
0,identifier,10.1234/gcd-2025
1,creators,Global Climate Data Team
2,title,2025 Global Climate Data
3,publisher,Open Research Repository (Teaching)
4,publicationYear,2025
5,resourceType,Dataset
6,description,A small teaching dataset containing fictional ...
